# 03 - XGBoost Classification

This notebook trains a binary XGBoost classifier to predict `flood_label` from the prepared environmental features and K-Means `cluster_label`. It keeps a stratified test set aside, compares a default-parameter baseline with a small cross-validated search, and saves the best model.

In [1]:
# Match the deployment environment so the saved classifier can be loaded by app.py.
%pip install -q xgboost==3.4.1

import xgboost
print("XGBoost version:", xgboost.__version__)
if xgboost.__version__ != "3.4.1":
    raise RuntimeError(
        "Restart the Colab runtime, then run this notebook from the beginning."
    )

from google.colab import drive
drive.mount('/content/drive')

XGBoost version: 3.4.1
Mounted at /content/drive


In [2]:
# Configure project paths and reproducibility settings once so all cells use the same dataset and model locations.
from pathlib import Path

PROJECT_ROOT_OVERRIDE = "/content/drive/My Drive/Flood Detection"
PROJECT_MARKERS = ("data", "models", "notebooks")


def find_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if all((candidate / marker).is_dir() for marker in PROJECT_MARKERS):
            return candidate
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not contain the project folders: {candidate}"
        )

    start_path = Path.cwd().resolve()
    for candidate in (start_path, *start_path.parents):
        if all((candidate / marker).is_dir() for marker in PROJECT_MARKERS):
            return candidate
    raise FileNotFoundError(
        f"Project root not found from kernel directory {start_path}. "
        "Set PROJECT_ROOT_OVERRIDE to a project path visible to this kernel. "
        "A remote kernel needs the project files mounted or copied into its filesystem."
    )


PROJECT_ROOT = find_project_root()
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "flood_data_processed.csv"
FEATURE_COLUMNS_PATH = PROJECT_ROOT / "models" / "feature_columns.json"
MODEL_PATH = PROJECT_ROOT / "models" / "xgb_flood_model.joblib"
TARGET_COLUMN = "flood_label"
TEST_SIZE = 0.20
RANDOM_STATE = 42
CV_FOLDS = 3

## Load and Validate the Processed Data

The feature list is read from the shared JSON contract to keep training and the serving app in the same column order. `flood_label` is the target and must not appear among model inputs.

In [3]:
# Load the clustered training table and enforce the shared predictor order and binary target contract.
import json
import joblib
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

if not PROCESSED_DATA_PATH.is_file():
    raise FileNotFoundError(f"Processed dataset not found: {PROCESSED_DATA_PATH}. Run notebooks 01 and 02 first.")
if not FEATURE_COLUMNS_PATH.is_file():
    raise FileNotFoundError(f"Feature contract not found: {FEATURE_COLUMNS_PATH}. Run notebook 02 first.")

with FEATURE_COLUMNS_PATH.open("r", encoding="utf-8") as feature_file:
    FEATURE_COLUMNS = json.load(feature_file)
if TARGET_COLUMN in FEATURE_COLUMNS:
    raise ValueError("flood_label is the target and must not be included in the model feature list.")

df = pd.read_csv(PROCESSED_DATA_PATH)
required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Processed dataset is missing required columns: {missing_columns}")
if df[required_columns].isna().any().any():
    raise ValueError("Model columns contain missing values. Clean them in notebook 01 before training.")

X = df[FEATURE_COLUMNS]
y = pd.to_numeric(df[TARGET_COLUMN], errors="raise").astype(int)
if set(y.unique()) != {0, 1}:
    raise ValueError(f"Expected both binary labels 0 and 1; found {sorted(y.unique().tolist())}.")

print(f"Dataset shape: {df.shape}")
print(f"Rows: {len(df)} | Features: {len(FEATURE_COLUMNS)}")
print("Feature order:", FEATURE_COLUMNS)
print("Label counts:", y.value_counts().sort_index().to_dict())

Dataset shape: (3000, 10)
Rows: 3000 | Features: 9
Feature order: ['rainfall', 'soil_moisture', 'humidity', 'ndvi', 'ndwi', 'elevation', 'slope', 'temperature', 'cluster_label']
Label counts: {0: 1538, 1: 1462}


## Create the Train/Test Split

Stratification preserves the flood/no-flood proportions in both partitions. The test set stays out of hyperparameter selection and is used only for the baseline check here; notebook 04 will provide the full evaluation.

In [4]:
# Split observations 80/20 with class stratification so the test set is not used to tune model parameters.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(X_train)} ({len(X_train) / len(X):.0%})")
print(f"Test rows: {len(X_test)} ({len(X_test) / len(X):.0%})")
print("Training label proportions:", y_train.value_counts(normalize=True).sort_index().round(3).to_dict())
print("Test label proportions:", y_test.value_counts(normalize=True).sort_index().round(3).to_dict())

Training rows: 2400 (80%)
Test rows: 600 (20%)
Training label proportions: {0: 0.512, 1: 0.488}
Test label proportions: {0: 0.513, 1: 0.487}


## Train a Default XGBoost Baseline

The baseline uses XGBClassifier defaults, with `random_state=42` set for reproducibility. Its test F1 is a simple reference point; the test data is not used to choose hyperparameters.

In [5]:
# Fit a baseline XGBoost classifier with default model settings and report its held-out F1 score.
baseline_model = XGBClassifier(random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)
baseline_test_f1 = f1_score(y_test, baseline_predictions)
print(f"Baseline test F1: {baseline_test_f1:.4f}")

Baseline test F1: 0.7752


## Search a Small Hyperparameter Grid

The grid tests 2 values for each of three settings (8 combinations total) with 3-fold stratified cross-validation, for 24 fits. F1 is used because it balances precision and recall for the positive flood class. XGBoost uses one worker per fit while GridSearchCV parallelizes candidates to avoid nested CPU oversubscription.

In [6]:
# Compare a small reproducible grid using training data only, stratified three-fold CV, and F1 scoring.
parameter_grid = {
    "max_depth": [3, 5],
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
}
cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE
)
search = GridSearchCV(
    estimator=XGBClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_grid=parameter_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    refit=False,
    return_train_score=False,
)
search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print(f"Best cross-validation F1: {search.best_score_:.4f}")

Best hyperparameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Best cross-validation F1: 0.7413


## Refit the Best Model and Save It

After cross-validation chooses the settings, this cell creates a fresh classifier and fits it once on the complete training partition. The held-out test rows remain excluded from fitting.

> **Upstream preprocessing note:** notebooks 01 and 02 currently fit the scaler and K-Means before this split. That means the test feature transformations have seen all rows. For a rigorous, leakage-free estimate, fit preprocessing and clustering within each training fold; interpret the current CV/test scores with that limitation in mind.

In [7]:
# Refit the selected configuration on all training rows and save the fitted classifier for app inference.
best_model = XGBClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1,
    **search.best_params_,
)
best_model.fit(X_train, y_train)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)

# Verify the saved artifact can be loaded and used in this runtime before downloading it.
verified_model = joblib.load(MODEL_PATH)
sample_probabilities = verified_model.predict_proba(X_test.iloc[:1])
if sample_probabilities.shape != (1, 2):
    raise RuntimeError(f"Unexpected predict_proba shape: {sample_probabilities.shape}")

print(f"Saved and verified model: {MODEL_PATH}")
print("XGBoost version:", xgboost.__version__)
print("Sample probabilities:", sample_probabilities[0].tolist())
print("Best hyperparameters:", search.best_params_)
print(f"Best cross-validation F1: {search.best_score_:.4f}")
print(f"Baseline test F1: {baseline_test_f1:.4f}")

from google.colab import files
files.download(str(MODEL_PATH))

Saved and verified model: /content/drive/My Drive/Flood Detection/models/xgb_flood_model.joblib
XGBoost version: 3.4.1
Sample probabilities: [0.2861045002937317, 0.7138954997062683]
Best hyperparameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Best cross-validation F1: 0.7413
Baseline test F1: 0.7752


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Classification Summary

The saved model is fitted on the training partition only. Use notebook 04 for the final hold-out evaluation and probability-threshold review.